# Critical Exponent Analysis

Extract the critical exponent ν from finite-size scaling data.
Target: ν = 1.000 ± 0.001 (Ising universality class in 2D).

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import numpy as np
import matplotlib.pyplot as plt

from qft_graph.analysis.critical import (
    finite_size_scaling_crossing,
    extract_nu,
    susceptibility_peak,
)
from qft_graph.analysis.visualization import plot_scaling_collapse

plt.style.use('dark_background')
%matplotlib inline

In [ ]:
# Load sweep results (from 03_train_colab.ipynb or scripts/sweep.py)
# Adjust path as needed
results_path = '../experiments/runs/colab_run/sweep_results.json'

try:
    with open(results_path) as f:
        sweep_data = json.load(f)
    print('Loaded sweep results')
    for L, data in sweep_data.items():
        print(f'  L={L}: {len(data["m2_values"])} coupling values')
except FileNotFoundError:
    print(f'No sweep results found at {results_path}')
    print('Run the coupling sweep in 03_train_colab.ipynb or scripts/sweep.py first.')

In [ ]:
# Prepare data for analysis
L_values = [int(L) for L in sweep_data.keys()]
xi_over_L = {}
for L_str, data in sweep_data.items():
    L = int(L_str)
    xi_over_L[L] = list(zip(data['m2_values'], data['xi_over_L']))

# Find critical coupling from xi/L crossing
m2_c, m2_c_err = finite_size_scaling_crossing(L_values, xi_over_L)
print(f'Critical coupling: m²_c = {m2_c:.4f} ± {m2_c_err:.4f}')

# Extract nu from scaling collapse
nu, nu_err = extract_nu(L_values, xi_over_L, m2_c)
print(f'Critical exponent: ν = {nu:.3f} ± {nu_err:.3f}')
print(f'Target (Ising 2D): ν = 1.000')

# Susceptibility peaks
for L_str, data in sweep_data.items():
    m2_peak, chi_max = susceptibility_peak(
        np.array(data['m2_values']),
        np.array(data['chis'])
    )
    print(f'  L={L_str}: χ peak at m²={m2_peak:.3f} (χ_max={chi_max:.1f})')

In [ ]:
# Scaling collapse plot
fig = plot_scaling_collapse(
    L_values, xi_over_L, m2_c, nu,
    title=f'Scaling Collapse: ν = {nu:.3f} ± {nu_err:.3f}'
)
plt.show()